# 05 — Results Comparison

Evaluate trained models on test set and generate thesis figures:
- WRMSSE comparison (GNN vs Baseline)
- 30,490 Item-Store series evaluation
- Visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
np.random.seed(42)
torch.manual_seed(42)

## 5.1 Prepare Test Data

In [ ]:
from src.data.loader import load_m5
from src.gnn.dataset import build_dataloaders

dfs = load_m5()

_, _, test_loader = build_dataloaders(
    sales_df=dfs['sales'],
    calendar_df=dfs['calendar'],
    prices_df=dfs['prices'],
    G=None,
    batch_size=8, stride=7
)

## 5.2 Evaluate on Test Set

In [ ]:
from src.gnn.evaluate import run_evaluation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results = run_evaluation(test_loader, G=None, device=device)

## 5.3 Generate Thesis Figures

In [ ]:
from src.viz.comparison_viz import generate_all_figures

figures = generate_all_figures(results)

## 5.4 Results Summary

In [ ]:
print('\n=== FINAL TEST RESULTS ===')
print(f'{"Model":<22} {"MSE":>10} {"MAE":>10} {"WRMSSE":>10}')
print('-' * 55)
for name, key in [('DTNetGNN', 'gnn_test'), ('IsolatedBaseline', 'baseline_test')]:
    m = results[key]
    print(f'{name:<22} {m["mse"]:>10.6f} {m["mae"]:>10.4f} {m["wrmsse"]:>10.4f}')

gnn_w = results['gnn_test']['wrmsse']
base_w = results['baseline_test']['wrmsse']
improvement = (base_w - gnn_w) / base_w * 100
print(f'\nGNN improvement over Baseline: {improvement:.1f}%')